In [ ]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install seaborn
%pip install imbalanced-learn
%pip install scikit-learn
%pip install joblib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Crop and fertilizer dataset.csv'

import pandas as pd

# Load the dataset
data = pd.read_csv(file_path)

# Display the first few rows of the dataset
print("First few rows of the dataset:")
print(data.head())

print("\nDataset information (column names, data types, and missing values):")
print(data.info())

In [ ]:
# Filter the dataset for rice-related data only
rice_data = data[data['Crop'].str.lower() == 'rice']

# Drop irrelevant columns
rice_data = rice_data.drop(columns=['District_Name', 'Soil_color', 'Link', 'Crop'])

# Check for missing values
missing_values = rice_data.isnull().sum()

# Check for duplicate rows
duplicates = rice_data.duplicated().sum()

print(rice_data.head())
print(rice_data.shape)




In [ ]:
# Encode the target variable (Fertilizer)
rice_data['Fertilizer_encoded'] = rice_data['Fertilizer'].astype('category').cat.codes

# Display the mapping of fertilizer categories
fertilizer_mapping = dict(enumerate(rice_data['Fertilizer'].astype('category').cat.categories))

fertilizer_mapping

In [ ]:
import matplotlib.pyplot as plt

# Check for outliers using boxplots
numerical_features = ['Nitrogen', 'Phosphorus', 'Potassium', 'pH', 'Rainfall', 'Temperature']

# Plot boxplots for each numerical feature to identify outliers
for feature in numerical_features:
    plt.figure()
    plt.boxplot(rice_data[feature])
    plt.title(f'Boxplot of {feature}')
    plt.xlabel(feature)
    plt.show()




In [ ]:
# Function to remove outliers using the IQR method
def remove_outliers_iqr(df, columns):
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Removing outliers
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    return df

# Apply the function to numerical features
numerical_features = ['Nitrogen', 'Phosphorus', 'Potassium', 'pH', 'Rainfall', 'Temperature']
rice_data_cleaned = remove_outliers_iqr(rice_data, numerical_features)

# Display the cleaned data after outlier removal
print("Rice Dataset After Removing Outliers\n", rice_data_cleaned)


In [ ]:
# Plot boxplots for each numerical feature after outlier removal
for feature in numerical_features:
    plt.figure()
    plt.boxplot(rice_data_cleaned[feature])
    plt.title(f'Boxplot of {feature} (Outliers Removed)')
    plt.xlabel(feature)
    plt.show()

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Features and target
X = rice_data_cleaned[['Nitrogen', 'Phosphorus', 'Potassium', 'pH', 'Rainfall', 'Temperature']]
y = rice_data_cleaned['Fertilizer_encoded']

# Split the data before applying Min-Max Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply SMOTE on the training set only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Check class distribution after SMOTE
from collections import Counter
print("Before SMOTE:", Counter(y_train))
print("After SMOTE:", Counter(y_train_smote))


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Apply Min-Max Scaling to the training data
X_train_scaled = scaler.fit_transform(X_train_smote)

# Apply the same transformation to the test data
X_test_scaled = scaler.transform(X_test)

# Check the scaled data
import pandas as pd
scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)

print("Min-Max Scaled Training Data\n", scaled_df)


In [ ]:
import pandas as pd

# Define the path where you want to save the file
file_path = "/content/drive/My Drive/final_preprocessed_dataset.csv"

# Convert the final preprocessed dataset to a DataFrame
final_preprocessed_df = pd.DataFrame(X_train_scaled, columns=X.columns)
final_preprocessed_df['Fertilizer_encoded'] = y_train_smote  # Adding target column back

# Save the dataset to Google Drive
final_preprocessed_df.to_csv(file_path, index=False)

print(f"Final preprocessed dataset saved successfully at: {file_path}")


In [ ]:
# Add Gaussian noise
def add_no(data, mean=0, std=0.2):
    noise = np.random.normal(mean, std, data.shape)
    noisy_data = data + noise
    return noisy_data


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Add  noise to the training data
X_train_rf = add_no(X_train_scaled, std=0.2)

# Train the Random Forest Classifier
rf_model_train = RandomForestClassifier(random_state=42)
rf_model_train.fit(X_train_rf, y_train_smote)

# Predictions on the test set
y_pred_rf_test = rf_model_train.predict(X_test_scaled)
y_pred_rf_train = rf_model_train.predict(X_train_rf)

# Evaluate the model on the test set
accuracy_rf_test = accuracy_score(y_test, y_pred_rf_test)
report_rf_test = classification_report(y_test, y_pred_rf_test)
conf_matrix_rf_test = confusion_matrix(y_test, y_pred_rf_test)

# Evaluate the model on the training set
accuracy_rf_train = accuracy_score(y_train_smote, y_pred_rf_train)
report_rf_train = classification_report(y_train_smote, y_pred_rf_train)
conf_matrix_rf_train = confusion_matrix(y_train_smote, y_pred_rf_train)

# Results for the test set
print("Random Forest Model Performance with Maximum Noise (Test Set):")
print(f"Accuracy: {accuracy_rf_test:.4f}")
print("\nClassification Report:\n", report_rf_test)
print("Confusion Matrix:\n", conf_matrix_rf_test)

# Results for the training set
print("\nRandom Forest Model Performance with Maximum Noise (Training Set):")
print(f"Accuracy: {accuracy_rf_train:.4f}")
print("\nClassification Report:\n", report_rf_train)
print("Confusion Matrix:\n", conf_matrix_rf_train)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Regularized Random Forest
rf_model_regularized = RandomForestClassifier(
    n_estimators=100,      # Number of trees
    max_depth=5,           # Limit tree depth to prevent overfitting
    min_samples_split=10,  # Minimum samples required to split a node
    min_samples_leaf=5,    # Minimum samples required at a leaf node
    max_features='sqrt',   # Limit the number of features considered for splitting
    random_state=42
)

# Train the model
rf_model_regularized.fit(X_train_scaled, y_train_smote)

# Predictions
y_pred_rf_test_hp = rf_model_regularized.predict(X_test_scaled)
y_pred_rf_train_hp = rf_model_regularized.predict(X_train_scaled)

# Evaluation
accuracy_rf_test_hp = accuracy_score(y_test, y_pred_rf_test_hp)
report_rf_test_hp = classification_report(y_test, y_pred_rf_test_hp)
conf_matrix_rf_test_hp = confusion_matrix(y_test, y_pred_rf_test_hp)

accuracy_rf_train_hp = accuracy_score(y_train_smote, y_pred_rf_train_hp)
report_rf_train_hp = classification_report(y_train_smote, y_pred_rf_train_hp)
conf_matrix_rf_train_hp = confusion_matrix(y_train_smote, y_pred_rf_train_hp)

# Display results
print("Random Forest Model Performance (Test Set):")
print(f"Accuracy: {accuracy_rf_test_hp:.4f}")
print("\nClassification Report:\n", report_rf_test_hp)
print("Confusion Matrix:\n", conf_matrix_rf_test_hp)

print("\nRandom Forest Model Performance (Training Set):")
print(f"Accuracy: {accuracy_rf_train_hp:.4f}")
print("\nClassification Report:\n", report_rf_train_hp)
print("Confusion Matrix:\n", conf_matrix_rf_train_hp)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Get predicted probabilities for the positive class
y_probs = rf_model_regularized.predict_proba(X_test_scaled)

# Compute confidence scores
confidence = []
for val in y_probs:
    confidence.append(np.max(val))

# Plot histogram of confidence scores
plt.figure(figsize=(8, 5))
plt.hist(confidence, bins=10, color='skyblue', edgecolor='black', alpha=0.7)
plt.title('Confidence Scores')
plt.xlabel('Predicted Probability')
plt.ylabel('Frequency')
plt.show()

# Compute percentiles for thresholding
list_threshold = []
for i in range(5, 95, 10):  # Extending up to the 90th percentile
    threshold = np.percentile(confidence, i)
    print(f'{i}th percentile: {threshold:.2f}')
    list_threshold.append(threshold)


In [ ]:
# 55th
lowones55th = []
for values in confidence:
    if values < 0.72:
        print(values)
        lowones55th.append(values)

plt.figure(figsize=(8, 5))
plt.hist(lowones55th, bins=10, color='skyblue', edgecolor='black', alpha=0.7)
plt.title('Low Confidence Predictions')
plt.xlabel('Confidence Score')
plt.ylabel('Frequency')
plt.show()

print(f'55th Number of low-confidence predictions: {len(lowones55th)}\n')

# 65th
lowones65th = []
for values in confidence:
    if values < 0.74:
        print(values)
        lowones65th.append(values)


plt.figure(figsize=(8, 5))
plt.hist(lowones65th, bins=10, color='skyblue', edgecolor='black', alpha=0.7)
plt.title('Low Confidence Predictions')
plt.xlabel('Confidence Score')
plt.ylabel('Frequency')
plt.show()

print(f'65th Number of low-confidence predictions: {len(lowones65th)}\n')

# 75th
lowones75th = []
for values in confidence:
    if values < 0.75:
        print(values)
        lowones75th.append(values)

plt.figure(figsize=(8, 5))
plt.hist(lowones75th, bins=10, color='skyblue', edgecolor='black', alpha=0.7)
plt.title('Low Confidence Predictions')
plt.xlabel('Confidence Score')
plt.ylabel('Frequency')
plt.show()

print(f'75th Number of low-confidence predictions: {len(lowones75th)}\n')

In [ ]:
print(f"Best Threshold (55th percentile): {np.percentile(confidence, 55):.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Given confusion matrices
conf_matrix_rf_test_hp = [[18, 3], [1, 32]]
conf_matrix_rf_train_hp = [[124, 8], [4, 128]]

# Plotting the confusion matrix for the test set
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_rf_test_hp, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Plotting the confusion matrix for the training set
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_rf_train_hp, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
plt.title('Confusion Matrix - Training Set')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
import joblib

# Save the trained Random Forest model
joblib.dump(rf_model_regularized, '/content/drive/MyDrive/fertilizer_rf_model.joblib')
print("Model saved to fertilizer_rf_model.joblib")

# Save the scaler (important for preprocessing user inputs)
joblib.dump(scaler, '/content/drive/MyDrive/fertilizer_Scaler.pkl')
print("Scaler saved to fertilizer_Scaler.pkl")

In [ ]:
import joblib

# Load the Random Forest model and the scaler
rf_model = joblib.load('/content/drive/MyDrive/fertilizer_rf_model.joblib')
scaler = joblib.load('/content/drive/MyDrive/fertilizer_Scaler.pkl')

# Fertilizer mapping (0 = NPK, 1 = Urea)
fertilizer_mapping = {0: '50:26:26 NPK', 1: 'Urea'}

def predict_fertilizer_rf(N, P, K, pH, Rainfall, Temperature):
    # Prepare input data
    user_input = [[N, P, K, pH, Rainfall, Temperature]]

    # Scale the input data
    scaled_input = scaler.transform(user_input)

    # Predict the fertilizer
    predicted_class = rf_model.predict(scaled_input)[0]
    recommended_fertilizer = fertilizer_mapping[predicted_class]

    return recommended_fertilizer

# User input
N = float(input("Enter Nitrogen (N) level: "))
P = float(input("Enter Phosphorus (P) level: "))
K = float(input("Enter Potassium (K) level: "))
pH = float(input("Enter pH level: "))
Rainfall = float(input("Enter Rainfall (mm): "))
Temperature = float(input("Enter Temperature (°C): "))

# Predict and display the fertilizer recommendation
fertilizer = predict_fertilizer_rf(N, P, K, pH, Rainfall, Temperature)
print(f"\nRecommended Fertilizer: {fertilizer}")